# **SETUP**

In [ ]:
import pathlib
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

ohe = OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exists").set_output(transform="pandas")

# **ONE-HOT-ENCODING CATEGORICALS**

note: dropping driver because of high cardinality

In [27]:
train_subset = train.select_dtypes("category").drop(columns=["Driver"])
test_subset = test.select_dtypes("category").drop(columns=["Driver"])
oofs = []

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    ohe.fit(train_subset[~is_val])
    oofs.append(ohe.transform(train_subset[is_val]))

oof_out = pd.concat(oofs).sort_index().astype("int8")
test_out = ohe.transform(test_subset).astype("int8")

# **EXPORT**

In [28]:
oof_out.to_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "oof.parquet")
test_out.to_parquet(PROJECT_ROOT / "data" / "features" / "002-one-hot-categoricals" / "test.parquet")